# Stage 4: Evaluate Text Variables

**Purpose**: Resolves formula-based text variables by substituting parameter values and evaluating expressions.

**Input**: `data/intermediate/OBRA CIVIL/OBRA CIVIL_stage3.json`  
**Output**: `data/intermediate/OBRA CIVIL/OBRA CIVIL_stage4.json`

## What this notebook does

BC3 text variables can contain:

- **Parameter placeholders**: `%A`, `$B` → replaced with actual parameter values
- **Conditional expressions**: `(%A=B)` → evaluates to 1 or 0
- **Array indexing**: `$L(a,%C)` → looks up values in multi-dimensional arrays
- **Mathematical formulas**: Translated to Python and evaluated

### Example transformation

```
Original:  "$L(%A,%B)" where L is a lookup table and A="a", B="c"
Resolved:  "Concrete type HE-20"
```

The notebook processes data in chunks (1000 items) to handle memory constraints with large datasets. Text variables are processed in two passes:
1. Direct parameter replacements
2. Resolution of inter-variable dependencies

In [12]:
!pip install ijson

In [ ]:
import os
import json
import ijson
from typing import Iterator, Dict, Any
import traceback
import time

# Sprint 17 (Task D1): the core text-variable resolution transform (both
# passes) now lives in synthetic.stage_runners as importable, pure, in-memory
# functions. The inline copies of the formula helpers — including a
# `translate_formula_to_python` that had drifted from
# utils.z_formula_processing — have been removed; the runner uses the variant
# that reproduces the committed golden stage4 byte-for-byte (see
# docs/synthetic/RESEARCH_LOG.md, Sprint 17). This notebook keeps its own
# chunked file-IO driver and delegates the transform to the extracted core.
from synthetic.stage_runners import process_text_variables, process_unresolved_variables

# Generate a structured filename
# Input: Source file path, stage number, output directory, and file extension.
# Output: A structured file name for the output file.
def generate_filename(input_file, stage, output_dir, extension="json"):
    """Generate a structured filename with stage and timestamp."""
    base_name = os.path.splitext(os.path.basename(input_file))[0]  # Get base name of the source file
    file_name = f"{base_name}_stage{stage}.{extension}"
    return os.path.join(output_dir, file_name)

# Create output directories dynamically within the source file's path
# Input: Source file path.
# Output: Path to the created output directory.

def create_output_dirs(input_file):
    """Create and return the output directory path within the source file's directory."""
    input_dir = os.path.dirname(input_file)
    output_dir = input_dir # Same directory
    os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists
    return output_dir

CHUNK_SIZE = 1000  # Adjust based on available memory

def read_json_in_chunks(filename: str, chunk_size: int) -> Iterator[Dict[str, Any]]:
    """Read large JSON file in chunks using manual file reading."""
    chunk = {}
    items_processed = 0
    
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
        total_items = len(data)
        print(f"Total items to process: {total_items}")
        
        for key, value in data.items():
            chunk[key] = value
            items_processed += 1
            
            if len(chunk) >= chunk_size:
                print(f"Yielding chunk of {len(chunk)} items. Progress: {items_processed}/{total_items}")
                yield chunk
                chunk = {}
        
        if chunk:
            print(f"Yielding final chunk of {len(chunk)} items. Total processed: {items_processed}")
            yield chunk

def process_chunk(chunk: Dict[str, Any], output_dir: str, chunk_num: int) -> str:
    """Process a single chunk of data."""
    print(f"Processing chunk {chunk_num} with {len(chunk)} items")
    transformed_data = process_text_variables(chunk)
    final_data = process_unresolved_variables(transformed_data)
    
    chunk_file = os.path.join(output_dir, f'chunk_{chunk_num}.json')
    with open(chunk_file, 'w', encoding='utf-8') as f:
        json.dump(final_data, f, ensure_ascii=False, indent=4)
    
    return chunk_file

def merge_chunks(chunk_files: list, output_file: str):
    """Merge processed chunks into final output."""
    print(f"Merging {len(chunk_files)} chunks")
    merged_data = {}
    
    for chunk_file in chunk_files:
        with open(chunk_file, 'r', encoding='utf-8') as f:
            chunk_data = json.load(f)
            merged_data.update(chunk_data)
        os.remove(chunk_file)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(merged_data, f, ensure_ascii=False, indent=4)
    print(f"Merged data contains {len(merged_data)} items")

def main(input_file: str):
    """Process large JSON file in chunks."""
    start_time = time.time()
    stage3_input = generate_filename(input_file, stage=3, output_dir=os.path.dirname(input_file))
    output_dir = create_output_dirs(input_file)
    chunk_files = []
    
    try:
        print(f"Reading from: {stage3_input}")
        for i, chunk in enumerate(read_json_in_chunks(stage3_input, CHUNK_SIZE), 1):
            chunk_file = process_chunk(chunk, output_dir, i)
            chunk_files.append(chunk_file)
            
        stage4_output = generate_filename(input_file, stage=4, output_dir=output_dir)
        merge_chunks(chunk_files, stage4_output)
        
        print(f"Processing completed in {time.time() - start_time:.2f} seconds")
        print(f"Output saved to: {stage4_output}")
        
    except Exception as e:
        print(f"Error during processing:")
        traceback.print_exc()

# Input Format:
# {
#     "item_key": {
#         "text_variables": {
#             "var_name": string or list,  # Original text with placeholders
#             ...
#         },
#         "parameters": {
#             "param_name": {
#                 "label": string,
#                 "values": [
#                     {"label": string, "value": string},
#                     ...
#                 ]
#             },
#             ...
#         }
#     },
#     ...
# }

# Output Format:
# {
#     "item_key": {
#         "text_variables": {
#             "var_name": {
#                 "original": string or list,  # Original text
#                 "replaced": string or list,  # Text with placeholders replaced
#                 "evaluated": string or list, # Final evaluated text
#                 "is_formula": boolean,       # Whether it contains formula operations
#                 "dependencies": list,        # Other text variables referenced
#                 "fully_processed": boolean   # Whether all replacements are complete
#             },
#             ...
#         },
#         "parameters": { ... }  # Original parameters structure
#     },
#     ...
# }

In [14]:
from utils import config

input_file = config.chapter_path("OBRA CIVIL")
main(input_file)

Reading from: /work/data/intermediate/OBRA CIVIL/OBRA CIVIL_stage3.json
Total items to process: 126938
Yielding chunk of 1000 items. Progress: 1000/126938
Processing chunk 1 with 1000 items
Yielding chunk of 1000 items. Progress: 2000/126938
Processing chunk 2 with 1000 items
Yielding chunk of 1000 items. Progress: 3000/126938
Processing chunk 3 with 1000 items
Yielding chunk of 1000 items. Progress: 4000/126938
Processing chunk 4 with 1000 items
Yielding chunk of 1000 items. Progress: 5000/126938
Processing chunk 5 with 1000 items
Yielding chunk of 1000 items. Progress: 6000/126938
Processing chunk 6 with 1000 items
Yielding chunk of 1000 items. Progress: 7000/126938
Processing chunk 7 with 1000 items
Yielding chunk of 1000 items. Progress: 8000/126938
Processing chunk 8 with 1000 items
Yielding chunk of 1000 items. Progress: 9000/126938
Processing chunk 9 with 1000 items
Yielding chunk of 1000 items. Progress: 10000/126938
Processing chunk 10 with 1000 items
Yielding chunk of 1000 it